In [ ]:
import wfdb
import numpy as np
import pandas as pd
import time
import os
import json

DATA_PATH = "../data/vitals_data"

In [2]:
TARGET_VITALS = [
    "HR",
    "RESP",
    "SpO2",
]
CONDITIONING_VITALS = [
    'ABP Sys',
    'ABP Dias',
    'NBP Sys',
    'NBP Dias',
    'PVC Rate per Minute',
]

In [ ]:
top_dir = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
sub_dir = np.arange(0, 10000)

for t in top_dir:
    os.makedirs(DATA_PATH, exist_ok=True)
    CHECKPOINT_PATH = f"{DATA_PATH}/checkpoint_p0{t}.json"
    OUTPUT_PATH = f"{DATA_PATH}/mimic_vitals_p0{t}.parquet"

    print(f"Exploring folder p0{t}...")

    # Load checkpoint if exists
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            checkpoint = json.load(f)
        processed_patients = set(checkpoint["processed_patients"])
        record_hit_rate    = checkpoint["record_hit_rate"]
        missing_vitals     = checkpoint["missing_vitals"]
        patients           = checkpoint["patients"]
        print(f"Resuming from checkpoint: {len(processed_patients)} patients already processed")
    else:
        processed_patients = set()
        record_hit_rate    = {"total": 0, "keep": 0, "missing": 0, "short": 0}
        missing_vitals     = {v: 0 for v in TARGET_VITALS}
        patients           = 0
        print("Starting fresh")
    
    if os.path.exists(OUTPUT_PATH):
        df = pd.read_parquet(OUTPUT_PATH)
        print(f"  Loaded existing data: {len(df):,} rows")
    else:
        df = None

    patients_since_save = 0

    for s in sub_dir:
        s = str(s).zfill(4)
        patient_id = f"p0{t}{s}"
        if patient_id in processed_patients:
            continue
        
        try:
            patient_records = wfdb.get_record_list(f"mimic3wdb-matched/1.0/p0{t}/{patient_id}")
        except:
            processed_patients.add(patient_id)
            patients_since_save += 1
            continue

        print(f"     p0{t}{s}: Ingesting numerics")
        numerics_records = [r for r in patient_records if r.endswith("n")]
        patients += 1

        for record_name in numerics_records:
            record_hit_rate["total"] += 1
            header = wfdb.rdheader(
                record_name,
                pn_dir=f"mimic3wdb-matched/1.0/p0{t}/{patient_id}"
            )

            if header.sig_name is None or not all(v in header.sig_name for v in TARGET_VITALS):
                record_hit_rate["missing"] += 1
                for vital in TARGET_VITALS:
                    if vital not in (header.sig_name or []):
                        missing_vitals[vital] += 1
                print("       Skipping: Missing vitals...")
                continue

            # Only download up to 24 hours of data (+10 minutes for buffer)
            max_samples = min(60 * (1440 + 10), header.sig_len)

            t0 = time.time()
            record = wfdb.rdrecord(
                record_name,
                pn_dir=f"mimic3wdb-matched/1.0/p0{t}/{patient_id}",
                channel_names=TARGET_VITALS+CONDITIONING_VITALS,
                physical=True,
                sampfrom=0,
                sampto=max_samples
            )
            print(f"       Downloaded in {time.time() - t0:.1f} seconds")

            rec_df = pd.DataFrame(record.p_signal, columns=record.sig_name)
            rec_df = rec_df.reindex(columns=TARGET_VITALS+CONDITIONING_VITALS)

            fs = record.fs
            samples_per_minute = round(fs * 60)
            rec_df["Minute"] = np.arange(len(rec_df)) // samples_per_minute

            # Remove first and last 5 minutes of record
            max_minute = rec_df["Minute"].max()
            rec_df = rec_df[(rec_df["Minute"] >= 5) & (rec_df["Minute"] <= max_minute - 5)]

            # Downsample to per minute if needed
            if samples_per_minute > 1:
                rec_df = rec_df.groupby("Minute")[TARGET_VITALS + CONDITIONING_VITALS].mean().reset_index()

            # Min: 2 hours of data per record
            if rec_df["Minute"].nunique() < 120:
                print(f"         Skipping: less than 2 hours of data")
                record_hit_rate["short"] += 1
                continue

            # Max: first 24 hours of data per record
            rec_df = rec_df[rec_df["Minute"] < (rec_df["Minute"].min() + 1440)]

            rec_df["Timestamp"] = pd.to_timedelta(rec_df["Minute"], unit="min")
            rec_df["Patient"] = patient_id
            rec_df["Record"] = record_name
            record_hit_rate["keep"] += 1

            print(f"         {len(rec_df)} rows")

            if df is None:
                df = rec_df.reset_index(drop=True)
            else:
                df = pd.concat([df, rec_df.reset_index(drop=True)])

        patients_since_save += 1
        processed_patients.add(patient_id)
        if patients_since_save >= 100:
            if df is not None:
                df = df.reindex(columns=["Patient", "Record", "Minute"] + TARGET_VITALS + CONDITIONING_VITALS).reset_index(drop=True)
                df.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
                print(f"Saved {len(df):,} rows to {OUTPUT_PATH}")
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump({
                    "processed_patients": list(processed_patients),
                    "record_hit_rate":    record_hit_rate,
                    "missing_vitals":     missing_vitals,
                    "patients":           patients
                }, f)
            print(f"  Checkpoint saved: {len(processed_patients)} patients, {len(df):,} rows")
            patients_since_save = 0

    if df is not None:
        df = df.reindex(columns=["Patient", "Record", "Minute"] + TARGET_VITALS + CONDITIONING_VITALS).reset_index(drop=True)
        df.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
        print(f"Saved {len(df):,} rows to {OUTPUT_PATH}")

    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({
            "processed_patients": list(processed_patients),
            "record_hit_rate":    record_hit_rate,
            "missing_vitals":     missing_vitals,
            "patients":           patients
        }, f)
    
    # Final stats
    print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")
    print(f"Patients: {patients}")
    print(f"Record hit rate:")
    for k, v in record_hit_rate.items():
        print(f"  {k}: {v}")

Exploring folder p00...
Resuming from checkpoint: 10000 patients already processed
  Loaded existing data: 2,261,955 rows
Saved 2,261,955 rows to ../data/vitals_data/mimic_vitals_p00.parquet
File size: 12.8 MB
Patients: 1244
Record hit rate:
  total: 2588
  keep: 1858
  missing: 483
  short: 247
Exploring folder p01...
Resuming from checkpoint: 10000 patients already processed
  Loaded existing data: 2,143,280 rows
Saved 2,143,280 rows to ../data/vitals_data/mimic_vitals_p01.parquet
File size: 12.3 MB
Patients: 1171
Record hit rate:
  total: 2571
  keep: 1760
  missing: 538
  short: 273
Exploring folder p02...
Resuming from checkpoint: 10000 patients already processed
  Loaded existing data: 1,990,471 rows
Saved 1,990,471 rows to ../data/vitals_data/mimic_vitals_p02.parquet
File size: 12.6 MB
Patients: 1178
Record hit rate:
  total: 2477
  keep: 1641
  missing: 645
  short: 191
Exploring folder p03...
Resuming from checkpoint: 10000 patients already processed
  Loaded existing data: 36